# RepairLab: original-model feasibility baseline

Generate one proposed repair for each of **24 development problems**. This is
a **single-response baseline without execution feedback**, not an interactive
agent evaluation, training run, or final held-out result. No weights are trained.

Qwen3-0.6B is the first feasibility candidate because this project's predecessor
already ran it on a free GPU. It is not a claim that this is the best coding model.
Near-zero success would tell us to reconsider difficulty or the starting model.

**Kaggle:** import this notebook, keep it private, enable Internet, select one GPU.
**Colab:** upload the notebook, select a GPU runtime. Run cells in order.
Check your live free quota before starting; this notebook does not read it.
The run has 24 prompts, one response per prompt, and at most 1,024 new tokens
per response. It does not use paid inference or require an API key.

Only public task data and a version fingerprint are embedded. There are no
private cases, reference repairs, or grader in this notebook. Model-generated
Python is saved as text and NEVER executed here. Download the output archive
and evaluate it locally with the command at the end.


In [ ]:
import os
import sys
import subprocess
if sys.version_info < (3, 10):
    raise RuntimeError("Use a Python 3.10+ GPU runtime")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers==5.2.0"])


In [ ]:
import json
TASKS = json.loads('[{"public_tests":[{"args":[3,3],"expected":true},{"args":[0,3],"expected":false}],"source":"def solve(quantity, maximum):\\n    return 1 <= quantity < maximum\\n","specification":"For integer quantity and maximum >= 1, return whether 1 <= quantity <= maximum.","task_id":"quantity-range"},{"public_tests":[{"args":[2,2,5],"expected":true},{"args":[6,2,5],"expected":false}],"source":"def solve(timestamp, start, end):\\n    return start < timestamp <= end\\n","specification":"For integers start <= end, return whether timestamp is in the CLOSED interval [start,end].","task_id":"closed-window"},{"public_tests":[{"args":[4,4],"expected":true},{"args":[5,4],"expected":false}],"source":"def solve(size, limit):\\n    return size < limit\\n","specification":"Given nonnegative integer sizes, accept an empty or nonempty payload iff its size does not exceed the limit.","task_id":"payload-limit"},{"public_tests":[{"args":[[true,false]],"expected":false},{"args":[[true]],"expected":true}],"source":"def solve(present):\\n    return any(present)\\n","specification":"Return True iff every element of the boolean list present is True. An empty list is valid.","task_id":"required-fields"},{"public_tests":[{"args":[200,[200,201]],"expected":true},{"args":[500,[200,201]],"expected":false}],"source":"def solve(status, allowed):\\n    return status not in allowed\\n","specification":"Return whether integer status equals any item in allowed (a list of integer status codes).","task_id":"allowed-status"},{"public_tests":[{"args":["abc",3],"expected":true},{"args":["ab",3],"expected":false}],"source":"def solve(text, minimum):\\n    return len(text) > minimum\\n","specification":"Return whether text has at least minimum characters. text is a string; minimum is a nonnegative integer.","task_id":"password-length"},{"public_tests":[{"args":[[]],"expected":false},{"args":[[1,2]],"expected":true},{"args":[[1,0]],"expected":false}],"source":"def solve(quantities):\\n    return all(x > 0 for x in quantities)\\n","specification":"Accept a list of integer quantities iff it is nonempty and EVERY quantity is strictly positive.","task_id":"positive-batch"},{"public_tests":[{"args":[null],"expected":true},{"args":["  "],"expected":false},{"args":["Ada"],"expected":true}],"source":"def solve(name):\\n    return bool(name) and bool(name.strip())\\n","specification":"An optional name is valid if it is None, or a string containing at least one non-whitespace character.","task_id":"optional-name"},{"public_tests":[{"args":[[-2,1,2]],"expected":3},{"args":[[]],"expected":0}],"source":"def solve(values):\\n    return sum(values)\\n","specification":"Return the sum of ONLY strictly positive integers in values; return 0 for an empty list.","task_id":"positive-total"},{"public_tests":[{"args":[[2,1,2]],"expected":[2,1]},{"args":[[]],"expected":[]}],"source":"def solve(values):\\n    return sorted(set(values))\\n","specification":"Remove duplicate integers from values, preserving the order of their FIRST appearance.","task_id":"stable-unique"},{"public_tests":[{"args":[[1,2,3],2],"expected":[[1,2],[3]]},{"args":[[],2],"expected":[]}],"source":"def solve(values, size):\\n    return [values[i:i + size] for i in range(0, len(values) - size + 1, size)]\\n","specification":"Split values into consecutive nonempty chunks of at most size elements. size >= 1. Keep any final partial chunk.","task_id":"chunk-items"},{"public_tests":[{"args":[[0,1,2,3],2,2],"expected":[2,3]},{"args":[[1],0,0],"expected":[]}],"source":"def solve(values, offset, limit):\\n    return values[offset:limit]\\n","specification":"Return values[offset:offset+limit]. offset and limit are nonnegative integers; preserve list order.","task_id":"page-slice"},{"public_tests":[{"args":[[1,2,1],1],"expected":0},{"args":[[1,2],3],"expected":-1}],"source":"def solve(values, target):\\n    result = -1\\n    for i, value in enumerate(values):\\n        if value == target:\\n            result = i\\n    return result\\n","specification":"Return the index of the FIRST occurrence of target in values, or -1 if absent.","task_id":"first-match"},{"public_tests":[{"args":[[null,0,false,"",1]],"expected":[0,false,"",1]}],"source":"def solve(values):\\n    return [x for x in values if x]\\n","specification":"Remove only None entries from values. Preserve zero, False, empty strings, and order.","task_id":"drop-missing"},{"public_tests":[{"args":[[1,2,-1]],"expected":[1,3,2]},{"args":[[]],"expected":[]}],"source":"def solve(values):\\n    result = []\\n    total = 0\\n    for value in values:\\n        total = value\\n        result.append(total)\\n    return result\\n","specification":"Return inclusive prefix sums: output[i] is sum(values[:i+1]). An empty input returns [].","task_id":"running-totals"},{"public_tests":[{"args":[[1,2],3,0],"expected":[]},{"args":[[1,2],3,2],"expected":[2,3]}],"source":"def solve(values, item, capacity):\\n    return (values + [item])[-capacity:]\\n","specification":"Append item to values and return the last capacity items. capacity >= 0; zero capacity returns []. Do not mutate values.","task_id":"bounded-queue"},{"public_tests":[{"args":[true,3,3],"expected":false},{"args":[true,2,3],"expected":true},{"args":[false,0,3],"expected":false}],"source":"def solve(transient, attempts, maximum):\\n    return transient and attempts <= maximum\\n","specification":"Return True iff transient is True and attempts < maximum. attempts counts completed attempts; both counts are nonnegative integers.","task_id":"retry-decision"},{"public_tests":[{"args":[2,2,5],"expected":5},{"args":[2,0,5],"expected":2}],"source":"def solve(base, attempt, cap):\\n    return max(cap, base * 2 ** attempt)\\n","specification":"Return min(cap, base * 2**attempt). base,cap,attempt are nonnegative integers; attempt 0 uses base.","task_id":"capped-backoff"},{"public_tests":[{"args":[5,2,3],"expected":true},{"args":[4,2,3],"expected":false}],"source":"def solve(now, created, ttl):\\n    return now > created + ttl\\n","specification":"Return True iff the cache entry is expired: now >= created + ttl. All inputs are integers; ttl >= 0.","task_id":"cache-expiry"},{"public_tests":[{"args":[5,3],"expected":6},{"args":[6,3],"expected":6},{"args":[0,3],"expected":0}],"source":"def solve(now, interval):\\n    return (now // interval) * interval\\n","specification":"Return the smallest multiple of interval that is >= now. now >= 0 and interval >= 1 are integers; a slot exactly at now is allowed.","task_id":"next-scheduled-slot"},{"public_tests":[{"args":[3,5],"expected":0},{"args":[5,3],"expected":2}],"source":"def solve(total, used):\\n    return total - used\\n","specification":"Return max(0, total-used). total and used are nonnegative integers.","task_id":"remaining-budget"},{"public_tests":[{"args":[[true,false,true]],"expected":1},{"args":[[]],"expected":0}],"source":"def solve(outcomes):\\n    return sum(outcomes)\\n","specification":"Given booleans outcomes, return how many initial True entries precede the first False, or len(outcomes) if none fail.","task_id":"first-failure"},{"public_tests":[{"args":[[false,true,false]],"expected":1},{"args":[[true,true]],"expected":0}],"source":"def solve(outcomes):\\n    count = 0\\n    for ok in outcomes:\\n        if not ok:\\n            count += 1\\n    return count\\n","specification":"Return the number of consecutive False outcomes at the END of the boolean list. True means success and resets the count.","task_id":"consecutive-failures"},{"public_tests":[{"args":[[1,2,3,4],3,2],"expected":2},{"args":[[],3,2],"expected":0}],"source":"def solve(events, now, window):\\n    return sum(now - window <= t <= now for t in events)\\n","specification":"Count integer event timestamps satisfying now-window < timestamp <= now. window >= 0. Exclude the lower boundary and future events.","task_id":"rate-window"}]')
MANIFEST_SHA256 = '9829fbbd7e8db746505bcd6fb95205b181107cd207c63e79b52dbb014f30c407'
assert len(TASKS) == 24
print('Public tasks:', len(TASKS))


In [ ]:
import importlib.metadata
import os
from pathlib import Path
import random
import time
from datetime import datetime, timezone
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3-0.6B"
REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
SEED = 7
MAX_NEW_TOKENS = 1024
SYSTEM = (
    "Repair the Python function to satisfy its specification. "
    "Return ONLY the complete corrected Python source, exactly one function "
    "named solve. Preserve its signature. Do not include Markdown, decorators, "
    "annotations, default arguments, examples, or explanations. Inputs and "
    "outputs are JSON values. Preserve output types and do not mutate inputs. "
    "You cannot execute code or use tools. You get one response."
)
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
OUTPUT = Path("repairlab-baseline-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
OUTPUT.mkdir(exist_ok=False)
(OUTPUT / "raw").mkdir()
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
metadata = {
    "protocol": "single-response-v1", "split": "development", "model": MODEL,
    "model_revision": REVISION, "adapter": None, "seed": SEED,
    "manifest_sha256": MANIFEST_SHA256, "system_message": SYSTEM,
    "decoding": {"do_sample": False, "num_beams": 1, "num_return_sequences": 1,
                 "max_new_tokens": MAX_NEW_TOKENS, "enable_thinking": False,
                 "max_input_tokens": 3072},
    "source_extraction": "strip-whitespace-and-one-enclosing-code-fence-v1",
    "gpu": torch.cuda.get_device_name(0), "dtype": str(dtype),
    "packages": {p: importlib.metadata.version(p) for p in
                 ("torch", "transformers", "tokenizers", "huggingface-hub", "safetensors")},
    "created_at": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT / "model-config.json").write_text(json.dumps(metadata, indent=2))
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, revision=REVISION, torch_dtype=dtype, trust_remote_code=False
).to("cuda:0").eval()
print("Loaded", MODEL, REVISION, "Output:", OUTPUT)


In [ ]:
# No exec/eval/import of model output. All patches remain text until local grading.
def extract_source(raw):
    text = raw.strip()
    lines = text.splitlines()
    if len(lines) >= 3 and lines[0] in ("```python", "```") and lines[-1] == "```":
        return "\n".join(lines[1:-1]) + "\n"
    return text + "\n"

started = time.monotonic()
with (OUTPUT / "predictions.jsonl").open("w") as predictions:
    for index, task in enumerate(TASKS):
        begin = time.monotonic()
        row = {"task_id": task["task_id"]}
        try:
            messages = [{"role": "system", "content": SYSTEM},
                        {"role": "user", "content": json.dumps(task, ensure_ascii=False)}]
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
            inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
            if inputs.input_ids.shape[1] > 3072:
                raise ValueError("Input exceeds the fixed 3072-token budget")
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, do_sample=False, num_beams=1, num_return_sequences=1,
                    max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.eos_token_id
                )
            completion = generated[0, inputs.input_ids.shape[1]:]
            raw = tokenizer.decode(completion, skip_special_tokens=True)
            filename = task["task_id"].replace("/", "-") + ".txt"
            (OUTPUT / "raw" / filename).write_text(raw)
            row.update(source=extract_source(raw), raw_output=raw,
                       input_tokens=int(inputs.input_ids.shape[1]),
                       output_tokens=int(completion.shape[0]))
        except Exception as exc:
            row["generation_error"] = type(exc).__name__ + ": " + str(exc)[:1000]
        row["elapsed_seconds"] = time.monotonic() - begin
        predictions.write(json.dumps(row, ensure_ascii=False) + "\n")
        predictions.flush()
        print(index + 1, "/", len(TASKS), task["task_id"],
              "ERROR" if "generation_error" in row else "saved; not yet graded", flush=True)
        if row.get("generation_error", "").startswith("OutOfMemoryError"):
            print("Stopping after GPU memory failure; do not retry unchanged settings.")
            break
(OUTPUT / "runtime.json").write_text(json.dumps({
    "elapsed_seconds": time.monotonic() - started,
    "peak_gpu_bytes": torch.cuda.max_memory_allocated(0),
    "note": "Inference only. No repair execution, correctness scores, or weight updates."
}, indent=2))


In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT), "zip", OUTPUT)
print("Download:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(archive))


## Grade locally

Download and extract the archive. From the local `repairlab` directory, with
Docker running:

```sh
python3 -B -m repairlab score-patches \
  --predictions /absolute/path/to/extracted/predictions.jsonl \
  --metadata /absolute/path/to/extracted/model-config.json \
  --output artifacts/baseline-001
```

All 24 tasks remain in the denominator, including missing, malformed, and
failed generations. Infrastructure failures stop scoring and must be diagnosed.
Scores on these development tasks guide our next experiment; they are not a
final test result. Do not train on them and later call their scores held-out.
